<a href="https://colab.research.google.com/github/bchgacor/Data-Transaksi-Restoran/blob/main/Data_Transaksi_Restoran.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROJEK DATA MINING - DATA CLEANSING RESTORAN
Kasus: Pembersihan & Pengayaan Data Transaksi Restoran

In [ ]:
import pandas as pd
import numpy as np

# Load dataset kotor dari Google Drive
# Pastikan nama file sesuai dengan file yang diunggah
file_path = 'Data Kotor Restoran.xlsx'
df = pd.read_excel(file_path)

print("--- DATASET SEBELUM PEMBERSIHAN ---")
print(f"Jumlah Baris & Kolom: {df.shape}")
display(df.head(10))

--- DATASET SEBELUM PEMBERSIHAN ---
Jumlah Baris & Kolom: (30, 7)


,ID_Pesanan,Menu,Jumlah,Harga,Pelanggan,Waktu_Pesanan,Metode_Pembayaran
0,ORD-001,Nasi Goreng Spesial,2,25000,Raditya Arya,2026-09-01 12:30:00,Cash
1,ORD-002,Ayam Bakar,1,30000,Margaretha Nessa,2026-09-01 12:35:00,transfer
2,ORD-003,Es Teh Manis,-1,5000,Billy,2026-09-01 12:40:00,CASH
3,ORD-004,Mie Goreng,2,Rp.22.000,RADITYA,2026-09-01 12:45:00,Qris
4,ORD-005,es jeruk,1,7000,Gilang,2026-09-01 13:00:00,QRIS
5,ORD-006,Sate Ayam 10 Tusuk,2,35000,FANIA,2026-09-01 13:15:00,Debit Card
6,ORD-007,Nasi Goreng,1000,20000,Rian,2026-09-01 13:20:00,Cash
7,ORD-008,NaN,1,15000,Margaretha Nessa,2026-09-02 11:00:00,Transfer
8,ORD-009,Ayam Penyet,dua,28000,Isa,2026-09-02 11:15:00,cash
9,ORD-010,Soto Ayam,1,20000,NaN,2026-09-02 11:30:00,QRIS


# Data Cleansing - Standarisasi Kolom Kunci & Teks (ID, Menu, Pelanggan, Pembayaran)
Mengatasi spasi tak beraturan, huruf kapital acak, nama missing atau UNKNOWN, serta inkonsistensi penulisan metode pembayaran.

In [ ]:
# A. Standarisasi Kode Pesanan (Primary Key)
df['ID_Pesanan'] = df['ID_Pesanan'].astype(str).str.strip().str.upper()

# B. Standarisasi Nama Menu & Handling Missing Values
df['Menu'] = df['Menu'].astype(str).str.strip().str.title()
df['Menu'] = df['Menu'].replace({'Nan': 'Tidak Diketahui'})

# C. Standarisasi Nama Pelanggan (Hapus spasi liar, Title Case, atasi UNKNOWN/NaN)
df['Pelanggan'] = df['Pelanggan'].astype(str).str.strip().str.title()
df['Pelanggan'] = df['Pelanggan'].replace({'Nan': 'Anonim', 'Unknown': 'Anonim', 'Null': 'Anonim'})

# D. Standarisasi Metode Pembayaran
df['Metode_Pembayaran'] = df['Metode_Pembayaran'].astype(str).str.strip().str.title()
df['Metode_Pembayaran'] = df['Metode_Pembayaran'].replace({
    'Nan': 'Cash',
    'Debit Card': 'Debit',
    'Qris': 'QRIS',
    'Qris': 'QRIS'
})

print("--- HASIL STANDARISASI KOLOM TEKS ---")
display(df[['ID_Pesanan', 'Menu', 'Pelanggan', 'Metode_Pembayaran']].head(10))

--- HASIL STANDARISASI KOLOM TEKS ---


,ID_Pesanan,Menu,Pelanggan,Metode_Pembayaran
0,ORD-001,Nasi Goreng Spesial,Raditya Arya,Cash
1,ORD-002,Ayam Bakar,Margaretha Nessa,Transfer
2,ORD-003,Es Teh Manis,Billy,Cash
3,ORD-004,Mie Goreng,Raditya,QRIS
4,ORD-005,Es Jeruk,Gilang,QRIS
5,ORD-006,Sate Ayam 10 Tusuk,Fania,Debit
6,ORD-007,Nasi Goreng,Rian,Cash
7,ORD-008,Tidak Diketahui,Margaretha Nessa,Transfer
8,ORD-009,Ayam Penyet,Isa,Cash
9,ORD-010,Soto Ayam,Anonim,QRIS


# Data Cleansing - Standarisasi & Pembersihan Angka (Jumlah & Harga)
Mengubah teks menjadi angka, menghapus format currency, mengatasi nilai negatif, serta mengoreksi outlier.

In [ ]:
# A. Perbaikan Kolom Jumlah (Kuantitas Porsi)
df['Jumlah'] = df['Jumlah'].astype(str).str.strip().str.lower()
df['Jumlah'] = df['Jumlah'].replace({'dua': '2'})  # Konversi teks angka
df['Jumlah'] = pd.to_numeric(df['Jumlah'], errors='coerce').fillna(1).astype(int)

# Penanganan Nilai Negatif & Outlier Ekstrem pada Jumlah
df['Jumlah'] = df['Jumlah'].apply(lambda x: abs(x) if x < 0 else (1 if x in [0, 1000] else x))

# B. Perbaikan Kolom Harga (Hapus simbol 'Rp', pemisah ribuan, dan penanganan Outlier)
def bersihkan_harga(val):
    if pd.isna(val):
        return np.nan
    s = str(val).lower().replace('rp.', '').replace('rp', '').replace('.', '').replace(',', '').strip()
    try:
        num = abs(float(s))
        if num == 250000:  # Koreksi typo outlier Nasi Goreng Spesial (250.000 -> 25.000)
            num = 25000
        return num
    except:
        return np.nan

df['Harga'] = df['Harga'].apply(bersihkan_harga)

print("--- HASIL PEMBERSIHAN NUMERIK ---")
display(df[['Jumlah', 'Harga']].head(10))

--- HASIL PEMBERSIHAN NUMERIK ---


,Jumlah,Harga
0,2,25000.0
1,1,30000.0
2,1,5000.0
3,2,22000.0
4,1,7000.0
5,2,35000.0
6,1,20000.0
7,1,15000.0
8,2,28000.0
9,1,20000.0


# Data Cleansing - Standarisasi Kolom Waktu Pesanan
Mereset dan merapikan jam/tanggal transaksi ke dalam format datetime terstandar.

In [ ]:
# Mengubah ke format datetime standar (YYYY-MM-DD HH:MM:SS)
df['Waktu_Pesanan'] = pd.to_datetime(df['Waktu_Pesanan'], errors='coerce')

print("--- HASIL STANDARISASI WAKTU ---")
display(df[['Waktu_Pesanan']].head(10))

--- HASIL STANDARISASI WAKTU ---


,Waktu_Pesanan
0,2026-09-01 12:30:00
1,2026-09-01 12:35:00
2,2026-09-01 12:40:00
3,2026-09-01 12:45:00
4,2026-09-01 13:00:00
5,2026-09-01 13:15:00
6,2026-09-01 13:20:00
7,2026-09-02 11:00:00
8,2026-09-02 11:15:00
9,2026-09-02 11:30:00


# Data Cleansing - Deduplikasi Data
Menyeleksi dan menghapus transaksi duplikat berdasarkan Primary Key (ID_Pesanan).

In [ ]:
jumlah_sebelum = len(df)
df = df.drop_duplicates(subset=['ID_Pesanan']).reset_index(drop=True)
jumlah_sesudah = len(df)

print(f"Data Berhasil Dideduplikasi: Menghapus {jumlah_sebelum - jumlah_sesudah} baris duplikat.")
print(f"Total Baris Saat Ini: {jumlah_sesudah}")

Data Berhasil Dideduplikasi: Menghapus 0 baris duplikat.
Total Baris Saat Ini: 30


# Data Enrichment (Pengayaan Data)
Menambahkan variabel bisnis baru (Total_Bayar, Kategori_Pesanan, dan Jam_Pesanan) untuk mendukung analisis bisnis/laporan keuangan.

In [ ]:
# A. Menambahkan Total Pembayaran (Jumlah x Harga)
df['Total_Bayar'] = df['Jumlah'] * df['Harga']

# B. Menambahkan Klasifikasi Kategori Transaksi
df['Kategori_Pesanan'] = np.where(df['Total_Bayar'] >= 50000, 'Besar', 'Reguler')

# C. Ekstraksi Jam Transaksi untuk Analisis Peak Hours
df['Jam_Pesanan'] = df['Waktu_Pesanan'].dt.hour

print("--- HASIL AKHIR DATASET SETELAH DATA ENRICHMENT ---")
display(df.head(10))

--- HASIL AKHIR DATASET SETELAH DATA ENRICHMENT ---


,ID_Pesanan,Menu,Jumlah,Harga,Pelanggan,Waktu_Pesanan,Metode_Pembayaran,Total_Bayar,Kategori_Pesanan,Jam_Pesanan
0,ORD-001,Nasi Goreng Spesial,2,25000.0,Raditya Arya,2026-09-01 12:30:00,Cash,50000.0,Besar,12.0
1,ORD-002,Ayam Bakar,1,30000.0,Margaretha Nessa,2026-09-01 12:35:00,Transfer,30000.0,Reguler,12.0
2,ORD-003,Es Teh Manis,1,5000.0,Billy,2026-09-01 12:40:00,Cash,5000.0,Reguler,12.0
3,ORD-004,Mie Goreng,2,22000.0,Raditya,2026-09-01 12:45:00,QRIS,44000.0,Reguler,12.0
4,ORD-005,Es Jeruk,1,7000.0,Gilang,2026-09-01 13:00:00,QRIS,7000.0,Reguler,13.0
5,ORD-006,Sate Ayam 10 Tusuk,2,35000.0,Fania,2026-09-01 13:15:00,Debit,70000.0,Besar,13.0
6,ORD-007,Nasi Goreng,1,20000.0,Rian,2026-09-01 13:20:00,Cash,20000.0,Reguler,13.0
7,ORD-008,Tidak Diketahui,1,15000.0,Margaretha Nessa,2026-09-02 11:00:00,Transfer,15000.0,Reguler,11.0
8,ORD-009,Ayam Penyet,2,28000.0,Isa,2026-09-02 11:15:00,Cash,56000.0,Besar,11.0
9,ORD-010,Soto Ayam,1,20000.0,Anonim,2026-09-02 11:30:00,QRIS,20000.0,Reguler,11.0


# Ekspor Dataset Cleaned

In [ ]:
df.to_excel('Cleaned_Data_Restoran.xlsx', index=False)
print("File 'Cleaned_Data_Restoran.xlsx' berhasil disimpan dan siap dipergunakan!")

File 'Cleaned_Data_Restoran.xlsx' berhasil disimpan dan siap dipergunakan!
